# 📦 Fechas + mini integrador: la app de delivery — **SOLUCIÓN**

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/soluciones/10_fechas_integrador_solucion.ipynb)

**Objetivos:** parsear fechas que llegan como texto, extraer componentes con `.dt`, medir demoras con `Timedelta`, agrupar en el tiempo con `resample` y cerrar con un pipeline completo estilo certamen.

🔎 **Apóyate en el visualizador:** [Fechas](https://mati3939.github.io/visualizador-numpy-pandas/#fechas) · [GroupBy y pivoteo](https://mati3939.github.io/visualizador-numpy-pandas/#groupby) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: la app de delivery penquista

Una app de delivery exporta sus pedidos con las fechas en formato chileno `DD-MM-AAAA`… como **texto**. Todo el análisis temporal (demoras, estacionalidad, crecimiento por comuna) depende de arreglar eso primero — el mismo punto de partida del **Ejercicio Integrador 2** del curso.

## 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 80
_offsets = rng.integers(9, 90, n)
_offsets[0] = 61   # un pedido el 01-05-2026 (ya verás por qué importa…)
_offsets[1] = 9    # y el más antiguo de verdad: 10-03-2026
_pedido = pd.to_datetime('2026-03-01') + pd.to_timedelta(_offsets, unit='D')
_entrega = _pedido + pd.to_timedelta(rng.integers(1, 11, n), unit='D')
pedidos = pd.DataFrame({
    'id_pedido': np.arange(1, n + 1),
    'fecha_pedido': _pedido.strftime('%d-%m-%Y'),    # ¡texto a propósito!
    'fecha_entrega': _entrega.strftime('%d-%m-%Y'),  # ¡texto a propósito!
    'comuna': rng.choice(['Concepción', 'Talcahuano', 'San Pedro'], n, p=[.5, .3, .2]),
    'monto': rng.integers(8_000, 45_000, n),
})
print(pedidos.dtypes)
pedidos.head()

## 1 · Calentamiento

El flujo completo de hoy está animado en [https://mati3939.github.io/visualizador-numpy-pandas/#fechas](https://mati3939.github.io/visualizador-numpy-pandas/#fechas): parsear → `.dt` → restar → `resample`.

### 1.1 El orden alfabético miente ⭐

Sin parsear nada todavía: guarda en `primera_texto` el mínimo de `fecha_pedido` tal como está (texto), y en `primera_real` el mínimo después de parsear con `pd.to_datetime(..., format='%d-%m-%Y')` (formateado de vuelta a texto con `.strftime('%d-%m-%Y')`). Compara.

<details><summary>💡 Pista (haz clic)</summary>

En texto, `'01-05'` &lt; `'02-03'` porque `'1'` &lt; `'2'` en el segundo carácter. El mes ni alcanza a opinar.

</details>

In [ ]:
primera_texto = pedidos['fecha_pedido'].min()
primera_real = (pd.to_datetime(pedidos['fecha_pedido'], format='%d-%m-%Y')
                .min().strftime('%d-%m-%Y'))
# el texto compara carácter a carácter: '01-05-2026' le gana a '02-03-2026'
# aunque mayo venga DESPUÉS de marzo

print('según el texto :', primera_texto)
print('en la realidad :', primera_real)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert primera_texto != primera_real, 'con formato DD-MM, el orden alfabético NO coincide con el cronológico'
assert primera_real.endswith('-03-2026'), 'el primer pedido real es de marzo'
print('✅ ¡Correcto!')

### 1.2 Parsear de una vez por todas ⭐

Convierte AMBAS columnas de fecha a `datetime` (formato `%d-%m-%Y`), sobrescribiéndolas en `pedidos`. Verifica los `dtypes`.

<details><summary>💡 Pista (haz clic)</summary>

`format='%d-%m-%Y'` le dice a pandas que el DÍA va primero — sin eso, un `03-04-2026` es ambiguo y puede terminar en el mes equivocado.

</details>

In [ ]:
pedidos['fecha_pedido'] = pd.to_datetime(pedidos['fecha_pedido'], format='%d-%m-%Y')
pedidos['fecha_entrega'] = pd.to_datetime(pedidos['fecha_entrega'], format='%d-%m-%Y')
# format explícito: sin él, pandas puede confundir día con mes (03-04, ¿marzo o abril?)

print(pedidos.dtypes)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert pedidos['fecha_pedido'].dtype.kind == 'M', 'fecha_pedido debe ser datetime64'
assert pedidos['fecha_entrega'].dtype.kind == 'M'
assert (pedidos['fecha_entrega'] > pedidos['fecha_pedido']).all()
print('✅ ¡Correcto!')

## 2 · Núcleo

### 2.1 Componentes con .dt ⭐⭐

Agrega a `pedidos` las columnas `mes` (número de mes del pedido) y `dia_semana` (nombre del día, con `day_name()`). Después calcula `dia_top`: el día de la semana con MÁS pedidos. Ojo con el idioma del resultado.

<details><summary>💡 Pista (haz clic)</summary>

Sí, `day_name()` responde 'Friday' aunque tu notebook esté en español — es el mismo «¿por qué me salió Friday?» de la guía de GroupBy.

</details>

In [ ]:
pedidos['mes'] = pedidos['fecha_pedido'].dt.month
pedidos['dia_semana'] = pedidos['fecha_pedido'].dt.day_name()  # sale en INGLÉS
dia_top = pedidos['dia_semana'].value_counts().idxmax()

print(pedidos['dia_semana'].value_counts())
print('día top:', dia_top)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert set(pedidos['mes'].unique()).issubset({3, 4, 5})
assert dia_top in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
                   'Saturday', 'Sunday'], 'day_name() habla inglés por defecto'
assert pedidos['dia_semana'].value_counts()[dia_top] == pedidos['dia_semana'].value_counts().max()
print('✅ ¡Correcto!')

### 2.2 ¿Cuánto tardamos en entregar? ⭐⭐

Crea `pedidos['demora_dias']`: los días entre pedido y entrega (resta de columnas + `.dt.days`). Luego guarda en `id_mas_lento` el `id_pedido` del pedido con mayor demora.

In [ ]:
pedidos['demora_dias'] = (pedidos['fecha_entrega'] - pedidos['fecha_pedido']).dt.days
id_mas_lento = pedidos.loc[pedidos['demora_dias'].idxmax(), 'id_pedido']
# la resta de datetimes da Timedelta ('3 days'); .dt.days lo vuelve entero

print(pedidos['demora_dias'].describe().round(1))
print('más lento:', id_mas_lento)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert pedidos['demora_dias'].dtype.kind == 'i', 'demora_dias debe ser entero (días)'
assert (pedidos['demora_dias'] >= 1).all()
assert pedidos.loc[pedidos['id_pedido'] == id_mas_lento, 'demora_dias'].iloc[0] == pedidos['demora_dias'].max()
print('✅ ¡Correcto!')

### 2.3 Un rango de fechas con máscara ⭐⭐

Filtra en `abril` los pedidos hechos en abril de 2026 usando una máscara con dos condiciones sobre `fecha_pedido` (desde el 1 de abril inclusive, hasta antes del 1 de mayo). Puedes comparar directo contra strings `'2026-04-01'` — pandas los entiende.

<details><summary>💡 Pista (haz clic)</summary>

El extremo derecho va EXCLUYENTE (`< '2026-05-01'`): así no dependes de si el mes tiene 30 o 31 días.

</details>

In [ ]:
abril = pedidos[(pedidos['fecha_pedido'] >= '2026-04-01') &
                (pedidos['fecha_pedido'] < '2026-05-01')]
# contra una columna datetime, el string ISO 'AAAA-MM-DD' se convierte solo

print(len(abril), 'pedidos en abril')

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert (abril['fecha_pedido'].dt.month == 4).all()
assert len(abril) == (pedidos['fecha_pedido'].dt.month == 4).sum()
print('✅ ¡Correcto!')

### 2.4 Ventas mensuales con resample ⭐⭐

Construye `mensual`: el monto TOTAL vendido por mes, usando `set_index('fecha_pedido')` + `resample('ME')`. Fíjate en las etiquetas del índice resultante: ¿qué día de cada mes aparece?

<details><summary>💡 Pista (haz clic)</summary>

La animación de resample del visualizador muestra el paso a paso exacto: https://mati3939.github.io/visualizador-numpy-pandas/#fechas (tarjeta «groupby en el tiempo»).

</details>

In [ ]:
mensual = pedidos.set_index('fecha_pedido').resample('ME')['monto'].sum()
# 'ME' etiqueta cada periodo con el FIN de mes (2026-03-31, no 2026-03-01);
# en pandas antiguos se escribía 'M'

print(mensual)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(mensual) == 3, 'marzo, abril y mayo'
assert mensual.sum() == pedidos['monto'].sum(), 'no puede perderse plata en la agrupación'
assert mensual.index[0].day == 31, 'ME etiqueta con el fin de mes'
print('✅ ¡Correcto!')

## 3 · Desafío integrador 🏁

Todo lo del semestre en una pregunta de gerencia.

### 3.1 ¿Qué comuna crece más? ⭐⭐⭐

Construye `panel`: una `pivot_table` con el monto total por **comuna (filas) × mes (columnas)**, sin NaN (`fill_value=0`). Luego calcula `crecimiento`: la diferencia entre el último mes (5) y el primero (3) para cada comuna, y guarda en `comuna_estrella` la comuna que más creció.

<details><summary>💡 Pista (haz clic)</summary>

Es el mismo pivote del notebook 08, con el mes que fabricaste en el 2.1 como columna. El pipeline completo: texto → datetime → .dt → pivot.

</details>

In [ ]:
panel = pd.pivot_table(pedidos, values='monto', index='comuna',
                       columns='mes', aggfunc='sum', fill_value=0)
crecimiento = panel[5] - panel[3]
comuna_estrella = crecimiento.idxmax()

print(panel)
print(crecimiento.sort_values(ascending=False))
print('comuna estrella:', comuna_estrella)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert panel.shape == (3, 3), '3 comunas × 3 meses'
assert panel.to_numpy().sum() == pedidos['monto'].sum()
assert comuna_estrella in panel.index
assert crecimiento[comuna_estrella] == crecimiento.max()
print('✅ ¡Correcto!')

## 🏁 Cierre de la batería

Si llegaste hasta acá resolviendo (no mirando las soluciones 😄), ya recorriste el semestre completo: arrays, máscaras, Series, DataFrames, gráficos, nulos, outliers, groupby, joins y fechas. El siguiente nivel es el **Boss final** del visualizador — un DataFrame corrupto te espera.

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **Fechas**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).